In [1]:
from glob import glob
from sklearn.feature_extraction.text import CountVectorizer
import json
import os
import re
import itertools
import random
from tqdm import tqdm
from multiprocess import Pool
from collections import defaultdict

def generate_trigrams(text):
    words = text.split()
    return list(zip(words, words[1:], words[2:]))

def skip_trigrams(text):
    trigrams = generate_trigrams(text)
    count = defaultdict(int)
    total = 0
    for t in trigrams:
        count[''.join(t)] += 1
        total += 1
    if len(count.keys()) < 3:
        return True
    for k, v in count.items():
        if (v / total) > 0.2:
            return True
    return False

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [1]:
import os
import json
import re
import random
from tqdm import tqdm
from collections import defaultdict
from sklearn.feature_extraction.text import CountVectorizer

REJECTED = [
    'terima kasih kerana menonton',
    'terima kasih',
    'thank you for watching',
]

def new_path(f):
    return f.replace('_processed/', '_processed_trim_moshi/').replace('.mp3', '.moshi')

def load_and_filter_json(file_path):
    """Load JSON dan filter teks sesuai kriteria."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Warning: gagal load {file_path}: {e}")
        return []
    
    filtered = []
    for obj in data:
        text = obj.get("text", "").strip()
        audio_path = obj.get("audio_path", "")
        speaker = obj.get("speaker", "")
        if not text or not audio_path or not speaker:
            continue
        
        # filter rejected phrases
        rt_ = re.sub('[^a-z ]+', '', text.lower()).strip()
        if rt_ in REJECTED:
            continue
        
        # filter banyak single-character words
        split = text.split()
        ones = [w for w in split if len(w) <= 1]
        if len(ones) / max(len(split),1) >= 0.5:
            continue
        
        # filter repetitive chars
        if any([(len(set(w)) / len(w)) < 0.3 for w in split if len(w) > 0]):
            continue
        
        # filter repetitive trigrams
        try:
            dense = CountVectorizer(ngram_range=(3,3)).fit_transform([text]).todense()
            if (dense > 3).sum() >= 1:
                continue
        except:
            pass
        
        # pastikan file audio ada
        if not os.path.exists(audio_path):
            continue
        if not os.path.exists(new_path(audio_path)):
            continue
        
        filtered.append({
            "audio": audio_path,
            "transcription": text,
            "speaker": speaker
        })
    return filtered

def make_permutations(filtered_data, max_samples_per_speaker=30):
    """Buat permutasi reference-target per speaker."""
    speakers = defaultdict(list)
    for item in filtered_data:
        speakers[item["speaker"]].append(item)
    
    all_pairs = []
    for speaker, items in speakers.items():
        pairs = []
        n = len(items)
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                pairs.append({
                    "reference_audio": items[i]["audio"],
                    "reference_text": items[i]["transcription"],
                    "target_audio": items[j]["audio"],
                    "target_text": items[j]["transcription"],
                })
        if pairs:
            all_pairs.extend(random.sample(pairs, min(len(pairs), max_samples_per_speaker)))
    return all_pairs

def loop(files, max_samples_per_speaker=100):
    """Loop semua file JSON, filter, dan buat permutasi."""
    files, _ = files
    all_data = []
    for file in tqdm(files):
        filtered = load_and_filter_json(file)
        if filtered:
            pairs = make_permutations(filtered, max_samples_per_speaker)
            all_data.extend(pairs)
    return all_data

In [2]:
import os
import glob
import json
from tqdm import tqdm

# folder berisi JSON
json_folder = "/home/nafis/code/text-to-speech/dia/mesolitica-dia/malaya-speech/preprocessing/convert-json"

# ambil semua JSON
json_files = glob.glob(os.path.join(json_folder, "*.json"))

# prepare input format untuk loop()
files_input = (json_files, None)  # fungsi loop() mengharapkan tuple (files, _)

# jalankan loop
all_pairs = loop(files_input, max_samples_per_speaker=100)

import pandas as pd

pd.DataFrame(all_pairs).to_parquet('coba.parquet')

# # simpan hasil permutasi ke file JSON
# output_file = "/path/to/permuted_data.json"
# with open(output_file, "w", encoding="utf-8") as f:
#     json.dump(all_pairs, f, ensure_ascii=False, indent=2)

print(f"total pairs: {len(all_pairs)}")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:22<00:00,  3.25s/it]

total pairs: 42600


In [3]:
import pandas as pd

pd.DataFrame(all_pairs).to_parquet('permutate.parquet')

In [4]:
from datasets import load_dataset

dataset = load_dataset("parquet", data_files={'train': 'permutate.parquet'})

Generating train split: 0 examples [00:00, ? examples/s]

In [17]:
dataset["train"][6000]

{'reference_audio': '/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs/Ind050_F_S_C_appl_1319.wav',
 'reference_text': 'Nggak bener.',
 'target_audio': '/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs/Ind050_F_S_C_appl_1285.wav',
 'target_text': 'Matikan AC selama satu setengah jam.'}

In [18]:
import IPython.display as ipd
from IPython.display import display

sample = dataset["train"][6000]

print("Reference text:", sample["reference_text"])
display(ipd.Audio(sample["reference_audio"]))

print("Target text:", sample["target_text"])
display(ipd.Audio(sample["target_audio"]))


Reference text: Nggak bener.


Target text: Matikan AC selama satu setengah jam.


In [11]:
from IPython.display import Audio

# Path ke file wav
audio_file = "/home/nafis/code/text-to-speech/data/data/dara/wavs/1586.wav"

# Memutar audio
Audio(audio_file)